In [16]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list

from __future__ import annotations

import os  # needed navigate the system to get the input data

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

### initiate extractor

In [17]:
# Instantiate the extractor
paramPath = '/host/d/Github/Osteosarcoma/radiomics_settings/MR_setting_image.yaml'
extractor = featureextractor.RadiomicsFeatureExtractor(paramPath)

print('Extraction parameters:\n\t', extractor.settings)
print('Enabled filters:\n\t', extractor.enabledImagetypes)
print('Enabled features:\n\t', extractor.enabledFeatures)

Extraction parameters:
	 {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': True, 'normalizeScale': 100, 'removeOutliers': None, 'resampledPixelSpacing': [1, 1, 1], 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 10, 'voxelArrayShift': 300, 'geometryTolerance': 0.0001}
Enabled filters:
	 {'Original': {}, 'LoG': {'sigma': [2.0, 4.0, 6.0]}, 'Wavelet': {}}
Enabled features:
	 {'shape': None, 'firstorder': None, 'glcm': ['Autocorrelation', 'JointAverage', 'ClusterProminence', 'ClusterShade', 'ClusterTendency', 'Contrast', 'Correlation', 'DifferenceAverage', 'DifferenceEntropy', 'DifferenceVariance', 'JointEnergy', 'JointEntropy', 'Imc1', 'Imc2', 'Idm', 'Idmn', 'Id', 'Idn', 'InverseVariance', 'MaximumProbability', 'SumEntropy', 'SumSquares'], 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None}


### define patient list

In [21]:
# patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx'
patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/cases_with_label_reader2.xlsx'
build = Build_list.Build(patient_list_file)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')
# show one example of the image and mask
print('patient set:', patient_set_list[0], 'patient index:', patient_index_list[0], 'label:', label_list[0], 'image path:', image_path_list[0], 'mask path:', mask_path_list[0])


Number of cases to process: 28
patient set: set_1 patient index: 1 label: 0 image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz mask path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label_reader2.nii.gz


### extract features

In [22]:
rows = []
out_excel = '/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_reader2.xlsx'

for i in range(0, len(patient_index_list)):
    img_p = image_path_list[i]
    msk_p = mask_path_list[i]
    cid = patient_index_list[i]
    patient_set = patient_set_list[i]

    print('i', i, ' image path:', img_p, 'mask path:', msk_p, 'patient set:', patient_set, 'patient index:', cid)

    if (not os.path.isfile(img_p)) or (not os.path.isfile(msk_p)):
        print('  [skip] missing file')
        continue

    try:
        result = extractor.execute(img_p, msk_p)
    except Exception as e:
        print(f'  [skip] extractor failed: {e}')
        continue

    # Keep only radiomics features (drop diagnostics)
    feats = {k: v for k, v in result.items() if not k.startswith("diagnostics_")}
    feats["Patient_set"] = patient_set
    feats["Patient_index"] = cid
    feats["Image_filepath"] = img_p
    feats["Mask_filepath"] = msk_p

    rows.append(feats)

    # Build dataframe and save after each successful case so interruption will not lose extracted data.
    df = pd.DataFrame(rows)
    front_cols = [c for c in ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"] if c in df.columns]
    other_cols = [c for c in df.columns if c not in front_cols]
    df = df[front_cols + other_cols]

    df.to_excel(out_excel, index=False)
    print(f'  [saved] {len(df)} cases -> {out_excel}')

print('Extraction loop finished.')

i 0  image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz mask path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label_reader2.nii.gz patient set: set_1 patient index: 1
  [saved] 1 cases -> /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_reader2.xlsx
i 1  image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/5/img.nii.gz mask path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/5/label_reader2.nii.gz patient set: set_1 patient index: 5
  [saved] 2 cases -> /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_reader2.xlsx
i 2  image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/7/img.nii.gz mask path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/7/label_reader2.nii.gz patient set: set_1 patient index: 7
  [saved] 3 cases -> /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_reader2.xlsx
i 3  image path: /host/e/D/Data/Habitats/Jishuitan/original_d

### normalize features

#### normalize for reader 1

In [23]:
### normalize features
df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements.xlsx')
### normalize features to [0,1]
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
non_feature_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]
df_features = df[feature_cols]
df_features_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=feature_cols)
df_scaled = pd.concat([df[non_feature_cols], df_features_scaled], axis=1)
df_scaled.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx', index=False)

In [24]:
feature_min = scaler.data_min_
feature_max = scaler.data_max_

# 通过radiomics_measurements_normalized.xlsx计算得到的feature_min和feature_max更新radiomics_features_list.xlsx中的对应列, 然后每一行是一个feature, 这样就知道每个feature的min和max值了
df_scaled = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
# 我首先需要通过df_scaled的columns来获取feature list
feature_list = df_scaled.columns.tolist()
feature_list = [f for f in feature_list if f not in non_feature_cols]
feature_table = pd.DataFrame({'feature_name': feature_list, 'feature_min': feature_min, 'feature_max': feature_max})
feature_table.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_features_list.xlsx', index=False)

#### normalize for reader 2, will use data_min and data_max from reader 1 normalization

In [25]:
# for each feature, get the scaler.data_min_ and data_max_ from radiomics_features_list.xlsx, and then use it for normalizaton
scale_df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_features_list.xlsx')
df_reader2 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_reader2.xlsx')
feature_names = [c for c in df_reader2.columns if c not in non_feature_cols]
for feature in feature_names:
    f_min = scale_df.loc[scale_df['feature_name'] == feature, 'feature_min'].values[0]
    f_max = scale_df.loc[scale_df['feature_name'] == feature, 'feature_max'].values[0]
    df_reader2[feature] = (df_reader2[feature] - f_min) / (f_max - f_min)
df_reader2.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized_reader2.xlsx', index=False)

